# T-MI-0-SUPP Time-Domain Multifrequency Support-Recovery Baseline

This notebook measures support recovery as the number of microphones changes in a noiseless, time-domain generated, multifrequency simulation. It uses 10 candidate sources on a full ring, 2 active sources, and a full circular microphone array.

## Setup

This is a support-recovery benchmark, not an angular-resolution benchmark. Sources are generated in the time domain with `quick_sim(mode="noise")`, then frequency bins at or above 100 Hz are retained.

In [ ]:
from pathlib import Path
import os
import sys


def find_repo_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / "src" / "cs_priors").exists():
            return path
    raise RuntimeError("Could not find repository root containing src/cs_priors")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

os.environ.setdefault("MPLCONFIGDIR", "/tmp/cs_priors_matplotlib")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, precision_recall_curve

from cs_priors.plotting.plot_complex import plot_matrices
from cs_priors.plotting.plot_geometry import plot_sim_geometry
from cs_priors.simulation.mixing_model import quick_sim
from cs_priors.solvers.freq_lasso import frequency_lasso_solve
from cs_priors.solvers.freq_group_lasso import frequency_group_lasso_solve
from cs_priors.solvers.freq_sparse_prior import sparse_prior_solve

In [ ]:
TAG = "T-MI-0-SUPP"
FIGURE_DIR = REPO_ROOT / "results" / "benchmarks" / TAG
FIGURE_DIR.mkdir(parents=True, exist_ok=True)


def save_figure(fig, stem: str):
    fig.savefig(FIGURE_DIR / f"{stem}.pdf", dpi=300, bbox_inches="tight")


MIC_COUNTS = np.array([2, 3, 4, 5, 6, 8, 10])
MICS_TO_PLOT = [2, 3, 4]
NUM_SOURCES = 10
NUM_ACTIVE = 2
MIN_FREQ_HZ = 100.0
EXPECTED_NUM_FREQUENCIES = 45
SAMPLING_RATE = 2000.0
DURATION = 0.05
AMPLITUDE = 1.0

MIC_RADIUS = 0.3
MIC_ANGLE_START = 0.0
MIC_ANGLE_SPAN = 2 * np.pi
SOURCE_DISTANCE = 1.5
SOURCE_ANGLE_START = 0.0
SOURCE_ANGLE_SPAN = 2 * np.pi

SEEDS = np.arange(15)

LASSO_ALPHA = 1e-8
GROUP_LASSO_ALPHA = 1e-8
LASSO_MAX_ITER = 1_000
GROUP_LASSO_MAX_ITER = 500
THRESHOLD_FRACTION = 0.10
TOP_K = NUM_ACTIVE
PLOT_FLOOR = 1e-16

METHOD_ORDER = ["r-LASSO", "Group LASSO", "Sparse Prior", "X_pinv"]
METHOD_COLORS = {
    "X_pinv": "tab:blue",
    "r-LASSO": "tab:orange",
    "Group LASSO": "tab:green",
    "Sparse Prior": "tab:red",
}
METHOD_LINESTYLES = {
    "X_pinv": ":",
    "r-LASSO": "-",
    "Group LASSO": "-",
    "Sparse Prior": "-",
}

In [ ]:
def make_simulation(num_mics: int, seed: int):
    return quick_sim(
        num_mics=int(num_mics),
        num_sources=NUM_SOURCES,
        num_active=NUM_ACTIVE,
        seed=int(seed),
        sampling_rate=SAMPLING_RATE,
        duration=DURATION,
        source_distance=SOURCE_DISTANCE,
        mic_radius=MIC_RADIUS,
        array_type="circular",
        mic_angle_start=MIC_ANGLE_START,
        mic_angle_span=MIC_ANGLE_SPAN,
        source_angle_start=SOURCE_ANGLE_START,
        source_angle_span=SOURCE_ANGLE_SPAN,
        mode="noise",
        amplitude=AMPLITUDE,
        min_freq_hz=MIN_FREQ_HZ,
        single_frequency_hz=None,
        sensor_snr_db=None,
        model_snr_db=None,
        inverse_method="mp",
    )


def solve_methods(sim) -> dict[str, np.ndarray]:
    X_pinv = sim.X_pinv
    return {
        "X_pinv": X_pinv,
        "r-LASSO": frequency_lasso_solve(
            sim.Y,
            sim.A,
            alpha=LASSO_ALPHA,
            max_iter=LASSO_MAX_ITER,
            X_start=X_pinv,
        ),
        "Group LASSO": frequency_group_lasso_solve(
            sim.Y,
            sim.A,
            alpha=GROUP_LASSO_ALPHA,
            grouping="none",
            max_iter=GROUP_LASSO_MAX_ITER,
            X_start=X_pinv,
        ),
        "Sparse Prior": sparse_prior_solve(
            X_pinv,
            sim.A,
            grouping="none",
        ),
    }


def support_labels(sim) -> np.ndarray:
    labels = np.zeros(NUM_SOURCES, dtype=int)
    labels[np.asarray(sim.active_indices, dtype=int)] = 1
    return labels


def source_scores(X_hat: np.ndarray) -> np.ndarray:
    return np.linalg.norm(X_hat, axis=1)


def threshold_predictions(scores: np.ndarray) -> np.ndarray:
    max_score = float(np.max(scores))
    if max_score <= 0.0:
        return np.zeros_like(scores, dtype=int)
    return (scores >= THRESHOLD_FRACTION * max_score).astype(int)


def top_k_predictions(scores: np.ndarray, k: int = TOP_K) -> np.ndarray:
    pred = np.zeros_like(scores, dtype=int)
    top = np.argsort(scores)[-k:]
    pred[top] = 1
    return pred


def mean_squared_difference(X_hat: np.ndarray, X_true: np.ndarray) -> float:
    return float(np.mean(np.abs(X_hat - X_true) ** 2))


def assert_simulation_contract(sim, num_mics: int):
    assert sim.A.shape == (int(num_mics), NUM_SOURCES, EXPECTED_NUM_FREQUENCIES)
    assert sim.A.shape[2] > 1
    assert np.all(sim.freqs >= MIN_FREQ_HZ)
    assert len(sim.active_indices) == NUM_ACTIVE
    assert np.allclose(sim.eta, 0.0)
    assert np.allclose(sim.delta, 0.0)


def run_case(num_mics: int, seed: int) -> tuple[list[dict], list[dict]]:
    sim = make_simulation(num_mics, seed)
    assert_simulation_contract(sim, num_mics)

    y_true = support_labels(sim)
    rows = []
    score_rows = []

    for method, X_hat in solve_methods(sim).items():
        scores = source_scores(X_hat)
        y_pred_threshold = threshold_predictions(scores)
        y_pred_top2 = top_k_predictions(scores)

        rows.append(
            {
                "tag": TAG,
                "num_mics": int(num_mics),
                "seed": int(seed),
                "num_frequencies": int(sim.A.shape[2]),
                "method": method,
                "active_indices": tuple(sim.active_indices),
                "f1_threshold_10_percent": f1_score(y_true, y_pred_threshold, zero_division=0),
                "f1_top2": f1_score(y_true, y_pred_top2, zero_division=0),
                "mean_squared_difference": mean_squared_difference(X_hat, sim.X),
            }
        )

        for source_idx, (label, score) in enumerate(zip(y_true, scores)):
            score_rows.append(
                {
                    "tag": TAG,
                    "num_mics": int(num_mics),
                    "seed": int(seed),
                    "method": method,
                    "source_idx": int(source_idx),
                    "is_active": int(label),
                    "score": float(score),
                }
            )

    return rows, score_rows

In [ ]:
sim_check = make_simulation(MIC_COUNTS[-1], SEEDS[0])
assert_simulation_contract(sim_check, MIC_COUNTS[-1])

fig, ax = plot_sim_geometry(sim_check, show=False)
ax.set_title("T-MI-0-SUPP simulation geometry")
save_figure(fig, "simulation_geometry")
plt.show()

In [ ]:
EXAMPLE_NUM_MICS = 3
EXAMPLE_SEED = 0

example_sim = make_simulation(EXAMPLE_NUM_MICS, EXAMPLE_SEED)
assert_simulation_contract(example_sim, EXAMPLE_NUM_MICS)
example_solutions = solve_methods(example_sim)

EXAMPLE_METHOD_ORDER = ["X_pinv", "r-LASSO", "Group LASSO", "Sparse Prior"]

print("Active source indices:", example_sim.active_indices)
print("Retained frequencies:", example_sim.freqs)

plot_matrices(
    [example_sim.X, *(example_solutions[method] for method in EXAMPLE_METHOD_ORDER)],
    ["X_true", *EXAMPLE_METHOD_ORDER],
    show_values=False,
    polar=True,
    dpi=90,
)
fig = plt.gcf()
save_figure(fig, "example_solution_matrices")
plt.show()

In [ ]:
metric_rows = []
score_rows = []

for num_mics in MIC_COUNTS:
    for seed in SEEDS:
        case_metric_rows, case_score_rows = run_case(int(num_mics), int(seed))
        metric_rows.extend(case_metric_rows)
        score_rows.extend(case_score_rows)

results_df = pd.DataFrame(metric_rows)
scores_df = pd.DataFrame(score_rows)

for df in (results_df, scores_df):
    df["method"] = pd.Categorical(df["method"], categories=METHOD_ORDER, ordered=True)

results_df = results_df.sort_values(["num_mics", "seed", "method"]).reset_index(drop=True)
scores_df = scores_df.sort_values(["num_mics", "seed", "method", "source_idx"]).reset_index(drop=True)

assert set(results_df["num_mics"]) == set(MIC_COUNTS)
assert set(results_df["num_frequencies"]) == {EXPECTED_NUM_FREQUENCIES}
assert (
    scores_df.groupby(["num_mics", "seed", "method"], observed=False)["is_active"].sum()
    == NUM_ACTIVE
).all()
results_df.head()

In [ ]:
summary_df = (
    results_df
    .groupby(["num_mics", "method"], observed=False)
    .agg(
        median_f1_threshold_10_percent=("f1_threshold_10_percent", "median"),
        q25_f1_threshold_10_percent=("f1_threshold_10_percent", lambda x: x.quantile(0.25)),
        q75_f1_threshold_10_percent=("f1_threshold_10_percent", lambda x: x.quantile(0.75)),
        median_f1_top2=("f1_top2", "median"),
        q25_f1_top2=("f1_top2", lambda x: x.quantile(0.25)),
        q75_f1_top2=("f1_top2", lambda x: x.quantile(0.75)),
        median_mean_squared_difference=("mean_squared_difference", "median"),
        q25_mean_squared_difference=("mean_squared_difference", lambda x: x.quantile(0.25)),
        q75_mean_squared_difference=("mean_squared_difference", lambda x: x.quantile(0.75)),
    )
    .reset_index()
)
summary_df

In [ ]:
def plot_metric_with_band(metric_stem: str, ylabel: str, title: str, ylim=None, yscale="linear"):
    fig, ax = plt.subplots(figsize=(7.0, 4.4))

    for method in METHOD_ORDER:
        method_df = summary_df[summary_df["method"] == method].sort_values("num_mics")
        x = method_df["num_mics"].to_numpy(dtype=float)
        median = method_df[f"median_{metric_stem}"].to_numpy(dtype=float)
        q25 = method_df[f"q25_{metric_stem}"].to_numpy(dtype=float)
        q75 = method_df[f"q75_{metric_stem}"].to_numpy(dtype=float)

        if yscale == "log":
            median = np.maximum(median, PLOT_FLOOR)
            q25 = np.maximum(q25, PLOT_FLOOR)
            q75 = np.maximum(q75, PLOT_FLOOR)

        ax.plot(
            x,
            median,
            marker="o",
            label=method,
            color=METHOD_COLORS[method],
            linestyle=METHOD_LINESTYLES[method],
        )
        ax.fill_between(x, q25, q75, color=METHOD_COLORS[method], alpha=0.18, linewidth=0)

    ax.set_xticks(MIC_COUNTS)
    ax.set_xlabel("Number of microphones")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_yscale(yscale)
    if ylim is not None:
        ax.set_ylim(*ylim)
    ax.grid(True, which="both", linewidth=0.5, alpha=0.35)
    ax.legend(frameon=False)
    return fig, ax

In [ ]:
fig, ax = plot_metric_with_band(
    "f1_threshold_10_percent",
    "F1-score",
    "Time-domain multifrequency support F1 with 10% score threshold",
    ylim=(-0.03, 1.03),
)
save_figure(fig, "f1_threshold_10_percent_vs_num_mics")
plt.show()

In [ ]:
fig, ax = plot_metric_with_band(
    "f1_top2",
    "F1-score",
    "Time-domain multifrequency top-2 support F1",
    ylim=(-0.03, 1.03),
)
save_figure(fig, "f1_top2_vs_num_mics")
plt.show()

In [ ]:
fig, ax = plot_metric_with_band(
    "mean_squared_difference",
    "Mean squared difference",
    "Time-domain multifrequency coefficient mean squared difference",
    yscale="log",
)
save_figure(fig, "mean_squared_difference_vs_num_mics")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12.0, 3.6), sharex=True, sharey=True)
axes = axes.ravel()

for ax, num_mics in zip(axes, MICS_TO_PLOT):
    mic_scores = scores_df[scores_df["num_mics"] == num_mics]

    for method in METHOD_ORDER:
        curve_df = mic_scores[mic_scores["method"] == method]
        precision, recall, _ = precision_recall_curve(
            curve_df["is_active"].to_numpy(dtype=int),
            curve_df["score"].to_numpy(dtype=float),
        )

        ax.plot(
            recall,
            precision,
            label=method,
            color=METHOD_COLORS[method],
            linestyle=METHOD_LINESTYLES[method],
            linewidth=1.9,
        )

    ax.set_title(f"M = {num_mics}")
    ax.set_xlabel("Recall")
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)
    ax.grid(True, linewidth=0.5, alpha=0.35)

axes[0].set_ylabel("Precision")

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=4, frameon=False)
fig.suptitle("Time-domain multifrequency precision-recall curves by microphone count", y=1.02)
fig.tight_layout()

save_figure(fig, "precision_recall_curves")
plt.show()

## Interpretation

This notebook uses time-domain generated broadband noise, then analyzes retained frequency bins at or above 100 Hz. Group LASSO and Sparse Prior intentionally use no frequency-group structure. PR/F1 measure support ranking and detection, while MSE and matrix plots measure coefficient reconstruction.